# Day 03: The cost of picking the wrong model

**Question:** for a very basic task, how much money and time do we waste by using a big model when a small one does the job just as well?

**The task:** route customer support messages for an online store into one of 5 categories. This is the kind of high-volume, simple job a real product runs thousands of times a day.

**The models:**

| Role | Model | Input $ / 1M tokens | Output $ / 1M tokens |
|---|---|---|---|
| "Good" (big) model | `gpt-4o` | $2.50 | $10.00 |
| Light model | `gpt-4o-mini` | $0.15 | $0.60 |

Prices: OpenAI standard tier, checked 2026-10-11 at https://developers.openai.com/api/docs/pricing. Re-check before class; prices change.

We measure three things for each model:
1. **Performance:** % of messages routed to the correct category
2. **Cost:** from the real token counts the API returns
3. **Latency:** seconds per request (median and p95)

In [ ]:
import statistics
import time

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
client = OpenAI()

BIG_MODEL = "gpt-4o"
LIGHT_MODEL = "gpt-4o-mini"

# USD per 1M tokens (checked 2026-10-11, https://developers.openai.com/api/docs/pricing)
PRICES = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
}

## 1. The problem: a very basic classification task

Each message has a correct label, so we can score the models.

In [3]:
CATEGORIES = ["billing", "shipping", "refund", "technical", "other"]

INSTRUCTIONS = """You route customer support messages for an online store.
Reply with exactly one word, the category, from this list:
billing   - payments, charges, invoices, coupons, payment methods
shipping  - delivery status, tracking, delays, address for delivery
refund    - returns, refunds, cancellations, damaged or wrong items
technical - website or app problems, login, errors, bugs
other     - anything else
Reply with the category only, in lowercase, no punctuation."""

TICKETS = [
    ("I was charged twice for my order #4521.", "billing"),
    ("Where is my package? It's been 6 days.", "shipping"),
    ("The app crashes every time I open my cart.", "technical"),
    ("I want to return these shoes, they don't fit.", "refund"),
    ("Do you have a physical store in Pune?", "other"),
    ("My UPI payment failed but money was deducted.", "billing"),
    ("Tracking link says 'label created' for 4 days now.", "shipping"),
    ("I can't log in, it says invalid OTP every time.", "technical"),
    ("The phone cover arrived broken. I want my money back.", "refund"),
    ("Can I get a GST invoice for my last purchase?", "billing"),
    ("Please cancel my order, I ordered by mistake.", "refund"),
    ("Can you deliver to a PO box address?", "shipping"),
    ("The checkout page shows a blank white screen.", "technical"),
    ("Do you sell gift cards?", "other"),
    ("My coupon code SAVE10 isn't being applied.", "billing"),
    ("Courier marked it delivered but I never got it.", "shipping"),
    ("Password reset email never arrives.", "technical"),
    ("I received a blue shirt instead of the black one I ordered.", "refund"),
    ("Are you hiring for warehouse jobs?", "other"),
    ("When will my order be shipped? I paid yesterday.", "shipping"),
]

print(f"{len(TICKETS)} tickets, categories: {CATEGORIES}")

20 tickets, categories: ['billing', 'shipping', 'refund', 'technical', 'other']


## 2. One function: ask a model, time it, count tokens, price it

In [ ]:
def cost_usd(model: str, input_tokens: int, output_tokens: int) -> float:
    p = PRICES[model]
    return input_tokens / 1_000_000 * p["input"] + output_tokens / 1_000_000 * p["output"]


def classify(model: str, message: str) -> dict:
    start = time.perf_counter()
    response = client.responses.create(
        model=model,
        instructions=INSTRUCTIONS,
        input=message,
        temperature=0,
        max_output_tokens=16,
    )
    seconds = time.perf_counter() - start
    usage = response.usage
    return {
        "answer": response.output_text.strip().lower().strip("."),
        "seconds": seconds,
        "input_tokens": usage.input_tokens,
        "output_tokens": usage.output_tokens,
        "cost": cost_usd(model, usage.input_tokens, usage.output_tokens),
    }


classify(LIGHT_MODEL, "My order hasn't arrived yet.")

{'answer': 'shipping',
 'seconds': 2.0375703340105247,
 'input_tokens': 113,
 'output_tokens': 2,
 'cost': 1.8149999999999997e-05}

## 3. Run both models on all 20 tickets

We send one warm-up call per model first (the first call also opens the connection, which would make that model look slower), then alternate between the models ticket by ticket so both see the same network and server conditions.

In [4]:
results = {BIG_MODEL: [], LIGHT_MODEL: []}

for model in results:
    classify(model, "warm-up")

for text, label in TICKETS:
    for model in results:
        r = classify(model, text)
        r["correct"] = r["answer"] == label
        results[model].append(r)

print(f"{'ticket':<52} {'label':<10} {BIG_MODEL:<12} {LIGHT_MODEL:<12}")
for i, (text, label) in enumerate(TICKETS):
    big, light = results[BIG_MODEL][i], results[LIGHT_MODEL][i]
    mark = lambda r: r["answer"] + ("" if r["correct"] else " ✗")
    print(f"{text[:50]:<52} {label:<10} {mark(big):<12} {mark(light):<12}")

ticket                                               label      gpt-4o       gpt-4o-mini 
I was charged twice for my order #4521.              billing    billing      billing     
Where is my package? It's been 6 days.               shipping   shipping     shipping    
The app crashes every time I open my cart.           technical  technical    technical   
I want to return these shoes, they don't fit.        refund     refund       refund      
Do you have a physical store in Pune?                other      other        other       
My UPI payment failed but money was deducted.        billing    billing      billing     
Tracking link says 'label created' for 4 days now.   shipping   shipping     shipping    
I can't log in, it says invalid OTP every time.      technical  technical    technical   
The phone cover arrived broken. I want my money ba   refund     refund       refund      
Can I get a GST invoice for my last purchase?        billing    billing      billing     
Please can

## 4. Scorecard: performance, cost, latency

In [5]:
def summarise(rows: list[dict]) -> dict:
    secs = sorted(r["seconds"] for r in rows)
    return {
        "accuracy": sum(r["correct"] for r in rows) / len(rows),
        "input_tokens": sum(r["input_tokens"] for r in rows),
        "output_tokens": sum(r["output_tokens"] for r in rows),
        "cost": sum(r["cost"] for r in rows),
        "median_s": statistics.median(secs),
        "p95_s": statistics.quantiles(secs, n=20)[18],
        "total_s": sum(secs),
    }


summary = {model: summarise(rows) for model, rows in results.items()}
big, light = summary[BIG_MODEL], summary[LIGHT_MODEL]

print(f"{'':<22} {BIG_MODEL:>14} {LIGHT_MODEL:>14}")
print(f"{'accuracy':<22} {big['accuracy']:>14.0%} {light['accuracy']:>14.0%}")
print(f"{'input tokens':<22} {big['input_tokens']:>14,} {light['input_tokens']:>14,}")
print(f"{'output tokens':<22} {big['output_tokens']:>14,} {light['output_tokens']:>14,}")
print(f"{'cost for 20 tickets':<22} {'$' + format(big['cost'], '.6f'):>14} {'$' + format(light['cost'], '.6f'):>14}")
print(f"{'median latency':<22} {big['median_s']:>13.2f}s {light['median_s']:>13.2f}s")
print(f"{'p95 latency':<22} {big['p95_s']:>13.2f}s {light['p95_s']:>13.2f}s")
print()
print(f"{BIG_MODEL} costs {big['cost'] / light['cost']:.1f}x more than {LIGHT_MODEL} for the same job.")

                               gpt-4o    gpt-4o-mini
accuracy                         100%           100%
input tokens                    2,340          2,340
output tokens                      40             40
cost for 20 tickets         $0.006250      $0.000375
median latency                  0.94s          0.91s
p95 latency                     1.69s          1.31s

gpt-4o costs 16.7x more than gpt-4o-mini for the same job.


## 5. Now scale it to a real product

20 tickets cost fractions of a cent, so the difference looks tiny. Production is where it hurts. Below we take the measured cost per ticket and multiply it out.

In [6]:
TICKETS_PER_MONTH = 1_000_000  # try 10_000, 100_000, 10_000_000

per_ticket_big = big["cost"] / len(TICKETS)
per_ticket_light = light["cost"] / len(TICKETS)

monthly_big = per_ticket_big * TICKETS_PER_MONTH
monthly_light = per_ticket_light * TICKETS_PER_MONTH
saved_month = monthly_big - monthly_light

print(f"At {TICKETS_PER_MONTH:,} tickets / month:")
print(f"  {BIG_MODEL:<12} ${monthly_big:>10,.2f} / month   (${monthly_big * 12:,.2f} / year)")
print(f"  {LIGHT_MODEL:<12} ${monthly_light:>10,.2f} / month   (${monthly_light * 12:,.2f} / year)")
print(f"  saved by the light model: ${saved_month:,.2f} / month, ${saved_month * 12:,.2f} / year "
      f"({saved_month / monthly_big:.0%} cheaper)")
print()
print(f"Accuracy: {BIG_MODEL} {big['accuracy']:.0%} vs {LIGHT_MODEL} {light['accuracy']:.0%}")
print(f"Median wait per ticket: {BIG_MODEL} {big['median_s']:.2f}s vs {LIGHT_MODEL} {light['median_s']:.2f}s")

At 1,000,000 tickets / month:
  gpt-4o       $    312.50 / month   ($3,750.00 / year)
  gpt-4o-mini  $     18.75 / month   ($225.00 / year)
  saved by the light model: $293.75 / month, $3,525.00 / year (94% cheaper)

Accuracy: gpt-4o 100% vs gpt-4o-mini 100%
Median wait per ticket: gpt-4o 0.94s vs gpt-4o-mini 0.91s


## 6. What to take away

- On a simple task, the big model is **not more useful**, only more expensive. Compare the accuracy row: if both pass your bar, the extra cost is pure waste.
- The cost gap comes straight from the price table (~16x here) because both models used about the same number of tokens.
- Latency changes from run to run (network, server load). Run the notebook twice and compare. Never trust a single timing.
- **Rule:** start with the smallest model that passes your tests. Only move up when it fails on examples you care about.
- The flip side: on a hard task (multi-step reasoning, messy documents) the small model may fail. Then the "cheap" model is the expensive one, because wrong answers cost you retries, support time and users.

**Try it:** add 5 tricky tickets (e.g. "I was charged for an order that never arrived", which could be billing, shipping or refund) and see whether the two models start to disagree.